# Проверка карточек моделей

Компактная проверка версий моделей, находящихся в эксплуатации.

**Флаги:** `1` — обнаружен дефект, `0` — дефект не обнаружен.

Логика сверена со схемой, сэмплом, актуальным скриптом и альтернативным скриптом той же проверки. Физическая доступность содержимого файлов и репозиториев не проверяется.

## 1. Импорты и настройки окружения

In [ ]:
import os
import re
import sys
from datetime import date, datetime
from functools import reduce
from pathlib import Path

SPARK_CLIENT = Path("/usr/sdp/current/spark3.5.1-client")

# Сохраняем тот же Spark-клиент, что использовался в исходном ноутбуке.
if SPARK_CLIENT.exists():
    os.environ.setdefault("SPARK_MAJOR_VERSION", "3.5.1")
    os.environ.setdefault("SPARK_HOME", str(SPARK_CLIENT))
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

    spark_python = str(SPARK_CLIENT / "python")
    if spark_python not in sys.path:
        sys.path.insert(0, spark_python)

    py4j_archives = sorted((SPARK_CLIENT / "python/lib").glob("py4j-*-src.zip"))
    if py4j_archives:
        py4j_path = str(py4j_archives[-1])
        if py4j_path not in sys.path:
            sys.path.insert(0, py4j_path)

from pyspark import SparkConf
from pyspark.sql import SparkSession, Window, functions as F

from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
END_DATE_INCLUSIVE = True
MAX_EXCEL_ROWS = 1_048_575

EMPTY_TEXT = {"", "null", "none", "nan", "-", "[]", "{}"}
NA_TEXT = {
    "$not_applicable$",
    "not applicable",
    "не применимо",
    "неприменимо",
    "n/a",
    "na",
}
NO_RESULT_TEXT = {"отсутствует", *NA_TEXT}

LOCATION_PATTERN = (
    r"(?i)(?:"
    r"(?:https?|hdfs|s3a?|abfss?|wasbs?|file|dbfs)://[^\s<>]+"
    r"|\\\\[^\s\\]+\\[^\s]+"
    r"|(?:^|\s)/(?:[^\s/]+/)+[^\s]*"
    r"|(?:^|\s)[a-z]:\\[^\s]+"
    r"|\b[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*(?:\.[a-z_][a-z0-9_]*)?\b"
    r")"
)

# Bitbucket должен быть именно в hostname URL, а не просто встречаться в тексте пути.
BITBUCKET_PATTERN = r"(?i)https?://[^/\s<>]*bitbucket[^/\s<>]*(?:/|$)[^\s<>]*"

## 2. Spark-сессия

In [ ]:
spark = SparkSession.getActiveSession()

if spark is None:
    conf = (
        SparkConf()
        .setAppName("model_card_audit")
        .setMaster("yarn")
        .set("spark.executor.cores", "2")
        .set("spark.executor.memory", "6g")
        .set("spark.executor.memoryOverhead", "1g")
        .set("spark.driver.maxResultSize", "4g")
        .set("spark.shuffle.service.enabled", "true")
        .set("spark.dynamicAllocation.enabled", "true")
        .set("spark.dynamicAllocation.initialExecutors", "4")
        .set("spark.dynamicAllocation.maxExecutors", "12")
        .set("spark.port.maxRetries", "150")
    )
    spark = (
        SparkSession.builder
        .config(conf=conf)
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

## 3. Параметры запуска

In [ ]:
AS_OF = date.today().isoformat()
OUTPUT_FILE = Path.cwd() / "model_card_audit.xlsx"

print("Дата среза:", AS_OF)
print("Файл:", OUTPUT_FILE)

## 4. Вспомогательные функции

In [ ]:
def normalized(col):
    return F.lower(F.trim(F.regexp_replace(col.cast("string"), "\u00a0", " ")))


def filled(col):
    value = normalized(col)
    empty_array = value.rlike(r'^\[\s*(?:null|""|\s|,)*\s*\]$')
    return F.coalesce((~value.isin(*EMPTY_TEXT)) & (~empty_array), F.lit(False))


def is_na(col):
    return F.coalesce(normalized(col).isin(*NA_TEXT), F.lit(False))


def required_filled(col):
    """Содержательное значение обязательного реквизита."""
    value = normalized(col)
    return filled(col) & F.coalesce(~value.isin(*NO_RESULT_TEXT), F.lit(False))


def result_present(col):
    return required_filled(col)


def true_value(col):
    return F.coalesce(normalized(col).isin("true", "1", "да"), F.lit(False))


def has_location(col):
    return filled(col) & F.coalesce(
        col.cast("string").rlike(LOCATION_PATTERN),
        F.lit(False),
    )


def has_bitbucket(col):
    return filled(col) & F.coalesce(
        col.cast("string").rlike(BITBUCKET_PATTERN),
        F.lit(False),
    )


def valid_significance(col):
    return F.coalesce(
        F.upper(F.trim(col.cast("string"))).isin("A", "B", "C", "D", "E"),
        F.lit(False),
    )


def read_latest(spark, table, keys, fields, as_of, current=True):
    """
    Последнее сохранённое состояние записи на ключ.

    current=True  — текущий бизнес-срез.
    current=False — последнее состояние каждого сохранённого события/связи,
                    без ограничения по END_DT.
    """
    full_name = f"{SOURCE_DB}.{table}"
    if not spark.catalog.tableExists(full_name):
        raise RuntimeError(f"Не найдена таблица: {full_name}")

    raw = spark.table(full_name)
    lower = [c.lower() for c in raw.columns]
    if len(lower) != len(set(lower)):
        raise RuntimeError(f"{full_name}: неоднозначные имена столбцов")
    raw = raw.toDF(*lower)

    required = set(keys + fields + ["start_dt"])
    missing = sorted(required - set(raw.columns))
    if missing:
        raise RuntimeError(f"{full_name}: отсутствуют поля: {', '.join(missing)}")

    meta = [
        c
        for c in ("start_dt", "end_dt", "ctl_datechange", "ctl_action")
        if c in raw.columns
    ]
    df = raw.select(*dict.fromkeys(keys + fields + meta))

    for name in df.columns:
        if name.endswith("_sid"):
            df = df.withColumn(name, F.col(name).cast("string"))

    bound = F.lit(as_of).cast("date")
    df = df.filter(F.col("start_dt").cast("date") <= bound)

    # Пустой ключ нельзя молча выбрасывать: иначе аудит скрывает дефект источника.
    bad_key = reduce(
        lambda a, b: a | b,
        [~required_filled(F.col(k)) for k in keys],
        F.lit(False),
    )
    if df.filter(bad_key).limit(1).count():
        raise RuntimeError(f"{full_name}: обнаружен пустой/служебный ключ {keys}")

    order = [F.col("start_dt").desc_nulls_last()]
    if "ctl_datechange" in df.columns:
        order.append(F.col("ctl_datechange").desc_nulls_last())

    # dense_rank + проверка дублей: не выбираем случайную строку при полном равенстве
    # временных признаков.
    w = Window.partitionBy(*keys).orderBy(*order)
    df = (
        df.withColumn("__rank", F.dense_rank().over(w))
        .filter(F.col("__rank") == 1)
        .drop("__rank")
        .dropDuplicates()
    )

    duplicates = df.groupBy(*keys).count().filter(F.col("count") > 1)
    if duplicates.limit(1).count():
        raise RuntimeError(
            f"{full_name}: несколько разных последних записей на ключ {keys}"
        )

    # D исключаем только после выбора последнего состояния.
    if "ctl_action" in df.columns:
        df = df.filter(
            F.coalesce(
                F.upper(F.trim(F.col("ctl_action"))) != "D",
                F.lit(True),
            )
        )

    if current and "end_dt" in df.columns:
        end_date = F.col("end_dt").cast("date")
        valid_end = (
            end_date >= bound
            if END_DATE_INCLUSIVE
            else end_date > bound
        )
        df = df.filter(end_date.isNull() | valid_end)

    return df.select(*dict.fromkeys(keys + fields))

## 5. Формирование отчёта

In [ ]:
def build_report(spark, as_of):
    """Строит одну строку проверки на model_ver_sid."""
    date.fromisoformat(as_of)
    spark.sparkContext.setLogLevel("ERROR")

    models = read_latest(
        spark,
        "t_model",
        ["model_sid"],
        ["model_code", "model_name"],
        as_of,
    )

    versions = read_latest(
        spark,
        "t_model_ver",
        ["model_ver_sid"],
        [
            "model_sid",
            "model_ver_num",
            "model_ver_name",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name",
            "model_ver_owner_block_name",
            "model_ver_owner_dprtmt_name",
            "model_ver_owner_cont_name",
            "model_ver_dev_block_name",
            "model_ver_dev_dprtmt_name",
            "model_ver_dev_dprtmt_cont_name",
            "model_ver_dev_start_fact_dttm",
            "model_ver_dev_end_fact_dttm",
            "model_ver_dev_report_sid",
            "model_ver_repstry_link_txt",
            "model_ver_mlstorage_branch_link_txt",
            "model_ver_montrg_last_rslt_name",
            "model_ver_montrg_last_sid",
        ],
        as_of,
    )

    analytics = read_latest(
        spark,
        "t_model_ver_anlt_dtl",
        ["model_ver_sid"],
        [
            "model_ver_prom_expl_flag",
            "model_ver_not_valid_flag",
            "model_ver_busn_proc_code",
            "model_ver_prom_actual_implm_first_dttm",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name",
            "model_ver_valid_last_rslt_name",
        ],
        as_of,
    )

    base = (
        versions.alias("v")
        .join(analytics.alias("a"), "model_ver_sid", "inner")
        .filter(true_value(F.col("a.model_ver_prom_expl_flag")))
        .select(
            "model_ver_sid",
            F.col("v.model_sid").alias("model_sid"),
            F.col("v.model_ver_num").alias("model_ver_num"),
            F.col("v.model_ver_name").alias("model_ver_name"),
            F.col("v.model_ver_signfcnt_ctgry_code").alias("model_ver_signfcnt_ctgry_code"),
            F.col("v.model_ver_signfcnt_lvl_name").alias("model_ver_signfcnt_lvl_name"),
            F.col("a.model_ver_signfcnt_ctgry_code").alias("__anlt_signfcnt_ctgry_code"),
            F.col("v.model_ver_owner_block_name").alias("model_ver_owner_block_name"),
            F.col("v.model_ver_owner_dprtmt_name").alias("model_ver_owner_dprtmt_name"),
            F.col("v.model_ver_owner_cont_name").alias("model_ver_owner_cont_name"),
            F.col("v.model_ver_dev_block_name").alias("model_ver_dev_block_name"),
            F.col("v.model_ver_dev_dprtmt_name").alias("model_ver_dev_dprtmt_name"),
            F.col("v.model_ver_dev_dprtmt_cont_name").alias("model_ver_dev_dprtmt_cont_name"),
            F.col("v.model_ver_dev_start_fact_dttm").alias("model_ver_dev_start_fact_dttm"),
            F.col("v.model_ver_dev_end_fact_dttm").alias("model_ver_dev_end_fact_dttm"),
            F.col("v.model_ver_dev_report_sid").alias("model_ver_dev_report_sid"),
            F.col("v.model_ver_repstry_link_txt").alias("model_ver_repstry_link_txt"),
            F.col("v.model_ver_mlstorage_branch_link_txt").alias("model_ver_mlstorage_branch_link_txt"),
            F.col("v.model_ver_montrg_last_rslt_name").alias("model_ver_montrg_last_rslt_name"),
            F.col("v.model_ver_montrg_last_sid").alias("model_ver_montrg_last_sid"),
            F.col("a.model_ver_not_valid_flag").alias("model_ver_not_valid_flag"),
            F.col("a.model_ver_busn_proc_code").alias("model_ver_busn_proc_code"),
            F.col("a.model_ver_prom_actual_implm_first_dttm").alias("expl_start_dttm"),
            F.col("a.model_ver_valid_last_rslt_name").alias("model_ver_valid_last_rslt_name"),
        )
        .join(models, "model_sid", "left")
    )

    duplicate_codes = (
        models.filter(required_filled(F.col("model_code")))
        .withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .groupBy("__code")
        .agg(F.countDistinct("model_sid").alias("__code_count"))
    )
    base = (
        base.withColumn("__code", F.upper(F.trim(F.col("model_code"))))
        .join(duplicate_codes, "__code", "left")
    )

    # ------------------------------------------------------------------
    # Валидация
    # ------------------------------------------------------------------
    validations = read_latest(
        spark,
        "t_valid",
        ["valid_sid"],
        [
            "model_ver_sid",
            "valid_rslt_name",
            "valid_end_fact_dttm",
            "valid_not_sample_data_flag",
        ],
        as_of,
        current=False,
    ).filter(required_filled(F.col("model_ver_sid")))

    # Для результата/данных берём только завершённые на дату среза валидации.
    completed_validations = validations.filter(
        required_filled(F.col("valid_end_fact_dttm"))
        & (F.col("valid_end_fact_dttm").cast("date") <= F.lit(as_of).cast("date"))
    )

    samples = read_latest(
        spark,
        "t_sample_data",
        ["sample_data_sid"],
        [
            "model_ver_sid",
            "valid_sid",
            "montrg_manual_rslt_sid",
            "montrg_auto_rslt_sid",
            "sample_data_type_name",
            "sample_data_addtnl_info_txt",
        ],
        as_of,
        current=False,
    )

    valid_data_by_sid = (
        samples.filter(
            required_filled(F.col("valid_sid"))
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("valid_sid")
        .dropDuplicates(["valid_sid"])
        .withColumn("__valid_data_direct", F.lit(1))
    )

    valid_data_by_version = (
        samples.filter(
            required_filled(F.col("model_ver_sid"))
            & normalized(F.col("sample_data_type_name")).contains("валидац")
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("model_ver_sid")
        .dropDuplicates(["model_ver_sid"])
        .withColumn("__valid_data_version", F.lit(1))
    )

    validation_stats = (
        completed_validations
        .join(valid_data_by_sid, "valid_sid", "left")
        .join(valid_data_by_version, "model_ver_sid", "left")
        .withColumn(
            "__result_present",
            result_present(F.col("valid_rslt_name")).cast("int"),
        )
        .withColumn(
            "__result_data_ok",
            (
                result_present(F.col("valid_rslt_name"))
                & (
                    true_value(F.col("valid_not_sample_data_flag"))
                    | (F.coalesce(F.col("__valid_data_direct"), F.lit(0)) == 1)
                    | (F.coalesce(F.col("__valid_data_version"), F.lit(0)) == 1)
                )
            ).cast("int"),
        )
        .groupBy("model_ver_sid")
        .agg(
            F.max("__result_present").alias("__validation_result_present"),
            F.max("__result_data_ok").alias("__validation_result_data_ok"),
        )
    )

    base = (
        base.join(validation_stats, "model_ver_sid", "left")
        .join(valid_data_by_version, "model_ver_sid", "left")
        .fillna(
            {
                "__validation_result_present": 0,
                "__validation_result_data_ok": 0,
                "__valid_data_version": 0,
            }
        )
        .withColumn(
            "validation_not_required",
            true_value(F.col("model_ver_not_valid_flag")).cast("int"),
        )
        .withColumn(
            "has_validation_result",
            (
                (F.col("__validation_result_present") == 1)
                | result_present(F.col("model_ver_valid_last_rslt_name"))
            ).cast("int"),
        )
        .withColumn(
            "has_validation_data_path",
            (
                (F.col("__validation_result_data_ok") == 1)
                | (F.col("__valid_data_version") == 1)
            ).cast("int"),
        )
    )

    # ------------------------------------------------------------------
    # Мониторинг
    # ------------------------------------------------------------------
    base = base.withColumn(
        "has_monitor_result",
        result_present(F.col("model_ver_montrg_last_rslt_name")).cast("int"),
    )

    auto_results = read_latest(
        spark,
        "t_montrg_auto_rslt",
        ["montrg_auto_rslt_sid"],
        [
            "montrg_auto_sid",
            "montrg_auto_rslt_name",
            "montrg_auto_rslt_valid_not_sample_data_flag",
        ],
        as_of,
        current=False,
    ).filter(required_filled(F.col("montrg_auto_sid")))

    manual_results = read_latest(
        spark,
        "t_montrg_manual_rslt",
        ["montrg_manual_rslt_sid"],
        [
            "montrg_manual_sid",
            "montrg_manual_rslt_name",
            "montrg_manual_rslt_valid_not_sample_data_flag",
        ],
        as_of,
        current=False,
    ).filter(required_filled(F.col("montrg_manual_sid")))

    auto_links = read_latest(
        spark,
        "t_model_ver_x_montrg_auto",
        ["model_ver_sid", "montrg_auto_sid"],
        [],
        as_of,
        current=False,
    )

    manual_links = read_latest(
        spark,
        "t_model_ver_x_montrg_manual",
        ["model_ver_sid", "montrg_manual_sid"],
        [],
        as_of,
        current=False,
    )

    auto_sample_path = (
        samples.filter(
            required_filled(F.col("montrg_auto_rslt_sid"))
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("montrg_auto_rslt_sid")
        .dropDuplicates(["montrg_auto_rslt_sid"])
        .withColumn("__auto_result_has_data", F.lit(1))
    )

    manual_sample_path = (
        samples.filter(
            required_filled(F.col("montrg_manual_rslt_sid"))
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("montrg_manual_rslt_sid")
        .dropDuplicates(["montrg_manual_rslt_sid"])
        .withColumn("__manual_result_has_data", F.lit(1))
    )

    auto_monitor_data = (
        auto_results
        .join(auto_sample_path, "montrg_auto_rslt_sid", "left")
        .filter(result_present(F.col("montrg_auto_rslt_name")))
        .withColumn(
            "__auto_result_data_ok",
            (
                true_value(F.col("montrg_auto_rslt_valid_not_sample_data_flag"))
                | (F.coalesce(F.col("__auto_result_has_data"), F.lit(0)) == 1)
            ).cast("int"),
        )
        .groupBy("montrg_auto_sid")
        .agg(F.max("__auto_result_data_ok").alias("__auto_monitor_data_ok"))
    )

    manual_monitor_data = (
        manual_results
        .join(manual_sample_path, "montrg_manual_rslt_sid", "left")
        .filter(result_present(F.col("montrg_manual_rslt_name")))
        .withColumn(
            "__manual_result_data_ok",
            (
                true_value(F.col("montrg_manual_rslt_valid_not_sample_data_flag"))
                | (F.coalesce(F.col("__manual_result_has_data"), F.lit(0)) == 1)
            ).cast("int"),
        )
        .groupBy("montrg_manual_sid")
        .agg(F.max("__manual_result_data_ok").alias("__manual_monitor_data_ok"))
    )

    auto_by_version = (
        auto_links
        .join(auto_monitor_data, "montrg_auto_sid", "left")
        .select(
            F.col("model_ver_sid").alias("__auto_model_ver_sid"),
            F.col("montrg_auto_sid").alias("__auto_monitor_sid"),
            F.coalesce(F.col("__auto_monitor_data_ok"), F.lit(0))
            .alias("__auto_monitor_data_ok"),
        )
    )

    manual_by_version = (
        manual_links
        .join(manual_monitor_data, "montrg_manual_sid", "left")
        .select(
            F.col("model_ver_sid").alias("__manual_model_ver_sid"),
            F.col("montrg_manual_sid").alias("__manual_monitor_sid"),
            F.coalesce(F.col("__manual_monitor_data_ok"), F.lit(0))
            .alias("__manual_monitor_data_ok"),
        )
    )

    version_monitor_data = (
        samples.filter(
            required_filled(F.col("model_ver_sid"))
            & normalized(F.col("sample_data_type_name")).rlike(r"монитор|стабил")
            & has_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("model_ver_sid")
        .dropDuplicates(["model_ver_sid"])
        .withColumn("__version_monitor_data", F.lit(1))
    )

    base = (
        base
        .join(
            auto_by_version,
            (F.col("model_ver_sid") == F.col("__auto_model_ver_sid"))
            & (F.col("model_ver_montrg_last_sid") == F.col("__auto_monitor_sid")),
            "left",
        )
        .join(
            manual_by_version,
            (F.col("model_ver_sid") == F.col("__manual_model_ver_sid"))
            & (F.col("model_ver_montrg_last_sid") == F.col("__manual_monitor_sid")),
            "left",
        )
        .join(version_monitor_data, "model_ver_sid", "left")
        .withColumn(
            "has_monitor_data_path",
            (
                (F.coalesce(F.col("__auto_monitor_data_ok"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__manual_monitor_data_ok"), F.lit(0)) == 1)
                | (F.coalesce(F.col("__version_monitor_data"), F.lit(0)) == 1)
            ).cast("int"),
        )
    )

    # ------------------------------------------------------------------
    # Бинарные флаги: 1 = дефект, 0 = дефект не выявлен
    # ------------------------------------------------------------------
    base = (
        base
        .withColumn("F01_MODEL_CODE", (~required_filled(F.col("model_code"))).cast("int"))
        .withColumn("F02_MODEL_NAME", (~required_filled(F.col("model_name"))).cast("int"))
        .withColumn(
            "F03_MODEL_CODE_DUP",
            (F.coalesce(F.col("__code_count"), F.lit(0)) > 1).cast("int"),
        )
        .withColumn("F04_VERSION_NUM", (~required_filled(F.col("model_ver_num"))).cast("int"))
        .withColumn("F05_VERSION_NAME", (~required_filled(F.col("model_ver_name"))).cast("int"))
        .withColumn(
            "F06_DEV_START",
            (~required_filled(F.col("model_ver_dev_start_fact_dttm"))).cast("int"),
        )
        .withColumn(
            "F07_DEV_END",
            (~required_filled(F.col("model_ver_dev_end_fact_dttm"))).cast("int"),
        )
        .withColumn(
            "F08_EXPL_START",
            (~required_filled(F.col("expl_start_dttm"))).cast("int"),
        )
        .withColumn(
            "F09_OWNER",
            reduce(
                lambda a, b: a | b,
                [
                    ~required_filled(F.col(name))
                    for name in (
                        "model_ver_owner_block_name",
                        "model_ver_owner_dprtmt_name",
                        "model_ver_owner_cont_name",
                    )
                ],
                F.lit(False),
            ).cast("int"),
        )
        .withColumn(
            "F10_DEVELOPER",
            reduce(
                lambda a, b: a | b,
                [
                    ~required_filled(F.col(name))
                    for name in (
                        "model_ver_dev_block_name",
                        "model_ver_dev_dprtmt_name",
                        "model_ver_dev_dprtmt_cont_name",
                    )
                ],
                F.lit(False),
            ).cast("int"),
        )
        .withColumn(
            "F11_BUSN_PROCESS",
            F.when(is_na(F.col("model_ver_busn_proc_code")), F.lit(0))
            .otherwise((~filled(F.col("model_ver_busn_proc_code"))).cast("int")),
        )
        .withColumn(
            "F12_SIGNIFICANCE",
            (~valid_significance(F.col("model_ver_signfcnt_ctgry_code"))).cast("int"),
        )
        .withColumn(
            "F13_DEV_REPORT",
            (~required_filled(F.col("model_ver_dev_report_sid"))).cast("int"),
        )
        .withColumn(
            "F14_BITBUCKET_CODE",
            (~(
                has_bitbucket(F.col("model_ver_repstry_link_txt"))
                | has_bitbucket(F.col("model_ver_mlstorage_branch_link_txt"))
            )).cast("int"),
        )
        .withColumn(
            "F15_VALIDATION_RESULT",
            (
                (F.col("validation_not_required") == 0)
                & (F.col("has_validation_result") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F16_MONITOR_RESULT",
            (
                required_filled(F.col("model_ver_montrg_last_sid"))
                & (F.col("has_monitor_result") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F17_VALIDATION_DATA",
            (
                (F.col("validation_not_required") == 0)
                & (F.col("has_validation_result") == 1)
                & (F.col("has_validation_data_path") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F18_MONITOR_DATA",
            (
                required_filled(F.col("model_ver_montrg_last_sid"))
                & (F.col("has_monitor_result") == 1)
                & (F.col("has_monitor_data_path") == 0)
            ).cast("int"),
        )
        .withColumn(
            "F19_SIGNIFICANCE_MISMATCH",
            (
                valid_significance(F.col("model_ver_signfcnt_ctgry_code"))
                & valid_significance(F.col("__anlt_signfcnt_ctgry_code"))
                & (
                    normalized(F.col("model_ver_signfcnt_ctgry_code"))
                    != normalized(F.col("__anlt_signfcnt_ctgry_code"))
                )
            ).cast("int"),
        )
    )

    flag_columns = [
        "F01_MODEL_CODE",
        "F02_MODEL_NAME",
        "F03_MODEL_CODE_DUP",
        "F04_VERSION_NUM",
        "F05_VERSION_NAME",
        "F06_DEV_START",
        "F07_DEV_END",
        "F08_EXPL_START",
        "F09_OWNER",
        "F10_DEVELOPER",
        "F11_BUSN_PROCESS",
        "F12_SIGNIFICANCE",
        "F13_DEV_REPORT",
        "F14_BITBUCKET_CODE",
        "F15_VALIDATION_RESULT",
        "F16_MONITOR_RESULT",
        "F17_VALIDATION_DATA",
        "F18_MONITOR_DATA",
        "F19_SIGNIFICANCE_MISMATCH",
    ]

    base = base.withColumn(
        "issue_count",
        reduce(lambda a, b: a + b, [F.col(c) for c in flag_columns], F.lit(0)),
    )

    report_columns = [
        "model_sid",
        "model_code",
        "model_name",
        "model_ver_sid",
        "model_ver_num",
        "model_ver_name",
        "model_ver_signfcnt_ctgry_code",
        "model_ver_signfcnt_lvl_name",
        "model_ver_owner_block_name",
        "model_ver_owner_dprtmt_name",
        "model_ver_owner_cont_name",
        "model_ver_dev_block_name",
        "model_ver_dev_dprtmt_name",
        "model_ver_dev_dprtmt_cont_name",
        "expl_start_dttm",
        "issue_count",
        *flag_columns,
    ]

    return base.select(*report_columns).orderBy("model_sid", "model_ver_sid")

## 6. Экспорт в Excel

In [ ]:
def excel_value(value):
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def write_excel(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    wb = Workbook(write_only=True)
    ws = wb.create_sheet("Проверка")
    ws.freeze_panes = "A2"

    headers = list(frame.columns)

    # В write-only режиме ширину задаём до первой append().
    for idx in range(1, len(headers) + 1):
        ws.column_dimensions[get_column_letter(idx)].width = 24

    header_cells = []
    for header in headers:
        cell = WriteOnlyCell(ws, value=header)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="24435A")
        header_cells.append(cell)
    ws.append(header_cells)

    row_count = 0
    for row in frame.toLocalIterator():
        if row_count >= MAX_EXCEL_ROWS:
            raise RuntimeError("Результат превышает лимит одного листа Excel")

        cells = []
        for raw in row:
            value = excel_value(raw)
            if isinstance(value, str):
                value = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", value)
                if len(value) > 32_767:
                    raise ValueError("Превышен лимит текста ячейки Excel")

            cell = WriteOnlyCell(ws, value=value)
            if isinstance(value, str):
                cell.data_type = "s"
            cells.append(cell)

        ws.append(cells)
        row_count += 1

    if headers:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{row_count + 1}"

    temp = path.with_name(path.stem + ".partial.xlsx")
    try:
        wb.save(temp)
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()

## 7. QA / self-check

Контрольный расчёт строится отдельно от `build_report()` и сверяет:

- точный периметр `model_ver_sid`;
- одну строку на версию и значения справочных полей;
- все 19 флагов;
- распределение категорий значимости;
- отсутствие размножения/потери строк после join.

При расхождении QA показывает проблемные `model_ver_sid` и останавливает экспорт.

In [ ]:
QA_FAIL_ON_MISMATCH = True
QA_SHOW_MISMATCHES = 50

QA_INFO_COLUMNS = [
    "model_sid",
    "model_code",
    "model_name",
    "model_ver_num",
    "model_ver_name",
    "model_ver_signfcnt_ctgry_code",
    "model_ver_signfcnt_lvl_name",
    "model_ver_owner_block_name",
    "model_ver_owner_dprtmt_name",
    "model_ver_owner_cont_name",
    "model_ver_dev_block_name",
    "model_ver_dev_dprtmt_name",
    "model_ver_dev_dprtmt_cont_name",
    "expl_start_dttm",
]


# QA-предикаты намеренно определены отдельно от основных helpers.
_QA_EMPTY = ["", "null", "none", "nan", "-", "[]", "{}"]
_QA_NA = [
    "$not_applicable$",
    "not applicable",
    "не применимо",
    "неприменимо",
    "n/a",
    "na",
]
_QA_NO_RESULT = ["отсутствует", *_QA_NA]


def _qa_norm(col):
    return F.lower(
        F.trim(
            F.regexp_replace(
                F.coalesce(col.cast("string"), F.lit("")),
                "\u00a0",
                " ",
            )
        )
    )


def _qa_filled(col):
    value = _qa_norm(col)
    empty_array = value.rlike(r'^\[\s*(?:null|""|\s|,)*\s*\]$')
    return (~value.isin(*_QA_EMPTY)) & (~empty_array)


def _qa_required(col):
    value = _qa_norm(col)
    return _qa_filled(col) & (~value.isin(*_QA_NO_RESULT))


def _qa_true(col):
    return _qa_norm(col).isin("true", "1", "да")


def _qa_is_na(col):
    return _qa_norm(col).isin(*_QA_NA)


def _qa_significance(col):
    return F.upper(F.trim(F.coalesce(col.cast("string"), F.lit("")))).isin(
        "A", "B", "C", "D", "E"
    )


def _qa_location(col):
    text = F.coalesce(col.cast("string"), F.lit(""))
    return _qa_filled(col) & text.rlike(LOCATION_PATTERN)


def _qa_bitbucket(col):
    text = F.coalesce(col.cast("string"), F.lit(""))
    return _qa_filled(col) & text.rlike(
        r"(?i)https?://[^/\s<>]*bitbucket[^/\s<>]*(?:/|$)[^\s<>]*"
    )


def _qa_prepare_table(spark, table, keys, fields):
    full_name = f"{SOURCE_DB}.{table}"
    if not spark.catalog.tableExists(full_name):
        raise RuntimeError(f"QA: не найдена таблица {full_name}")

    raw = spark.table(full_name)
    names = [c.lower() for c in raw.columns]
    if len(names) != len(set(names)):
        raise RuntimeError(f"QA: {full_name}: неоднозначные имена столбцов")

    raw = raw.toDF(*names)

    required = set(keys + fields + ["start_dt"])
    missing = sorted(required - set(raw.columns))
    if missing:
        raise RuntimeError(
            f"QA: {full_name}: отсутствуют поля: {', '.join(missing)}"
        )

    meta = [
        c
        for c in ("start_dt", "end_dt", "ctl_datechange", "ctl_action")
        if c in raw.columns
    ]
    df = raw.select(*dict.fromkeys(keys + fields + meta))

    for name in df.columns:
        if name.endswith("_sid"):
            df = df.withColumn(name, F.col(name).cast("string"))

    return df


def _qa_latest_ranked(df, keys, as_of):
    """Последнее техническое состояние на ключ, без использования read_latest()."""
    bound = F.lit(as_of).cast("date")
    df = df.filter(F.col("start_dt").cast("date") <= bound)

    bad_key = reduce(
        lambda a, b: a | b,
        [~_qa_required(F.col(k)) for k in keys],
        F.lit(False),
    )
    if df.filter(bad_key).limit(1).count():
        raise RuntimeError(f"QA: обнаружен пустой/служебный ключ {keys}")

    ordering = [F.col("start_dt").desc_nulls_last()]
    if "ctl_datechange" in df.columns:
        ordering.append(F.col("ctl_datechange").desc_nulls_last())

    w = Window.partitionBy(*keys).orderBy(*ordering)
    ranked = (
        df.withColumn("__qa_rank", F.dense_rank().over(w))
        .filter(F.col("__qa_rank") == 1)
        .drop("__qa_rank")
        .dropDuplicates()
    )

    ambiguous = (
        ranked.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )
    if ambiguous:
        raise RuntimeError(
            f"QA: несколько разных последних состояний на ключ {keys}"
        )

    if "ctl_action" in ranked.columns:
        ranked = ranked.filter(
            F.coalesce(
                F.upper(F.trim(F.col("ctl_action"))) != "D",
                F.lit(True),
            )
        )
    return ranked


def _qa_current(spark, table, keys, fields, as_of):
    """
    Независимый текущий срез:
    сначала бизнес-граница END_DT, затем последнее техническое состояние.
    """
    df = _qa_prepare_table(spark, table, keys, fields)
    bound = F.lit(as_of).cast("date")

    if "end_dt" in df.columns:
        end_date = F.col("end_dt").cast("date")
        valid_end = (
            end_date >= bound
            if END_DATE_INCLUSIVE
            else end_date > bound
        )
        df = df.filter(end_date.isNull() | valid_end)

    df = _qa_latest_ranked(df, keys, as_of)
    return df.select(*dict.fromkeys(keys + fields))


def _qa_event(spark, table, keys, fields, as_of):
    """Независимое последнее состояние каждого сохранённого события."""
    df = _qa_prepare_table(spark, table, keys, fields)
    df = _qa_latest_ranked(df, keys, as_of)
    return df.select(*dict.fromkeys(keys + fields))


def build_qa_control(spark, as_of):
    """
    Независимо пересчитывает ожидаемый периметр, справочные значения и все флаги.
    Возвращает одну строку на model_ver_sid.
    """
    models = _qa_current(
        spark,
        "t_model",
        ["model_sid"],
        ["model_code", "model_name"],
        as_of,
    )

    versions = _qa_current(
        spark,
        "t_model_ver",
        ["model_ver_sid"],
        [
            "model_sid",
            "model_ver_num",
            "model_ver_name",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_signfcnt_lvl_name",
            "model_ver_owner_block_name",
            "model_ver_owner_dprtmt_name",
            "model_ver_owner_cont_name",
            "model_ver_dev_block_name",
            "model_ver_dev_dprtmt_name",
            "model_ver_dev_dprtmt_cont_name",
            "model_ver_dev_start_fact_dttm",
            "model_ver_dev_end_fact_dttm",
            "model_ver_dev_report_sid",
            "model_ver_repstry_link_txt",
            "model_ver_mlstorage_branch_link_txt",
            "model_ver_montrg_last_rslt_name",
            "model_ver_montrg_last_sid",
        ],
        as_of,
    )

    analytics = _qa_current(
        spark,
        "t_model_ver_anlt_dtl",
        ["model_ver_sid"],
        [
            "model_ver_prom_expl_flag",
            "model_ver_not_valid_flag",
            "model_ver_busn_proc_code",
            "model_ver_prom_actual_implm_first_dttm",
            "model_ver_signfcnt_ctgry_code",
            "model_ver_valid_last_rslt_name",
        ],
        as_of,
    )

    duplicate_codes = (
        models.filter(_qa_required(F.col("model_code")))
        .withColumn("__qa_code", F.upper(F.trim(F.col("model_code"))))
        .groupBy("__qa_code")
        .agg(F.countDistinct("model_sid").alias("__qa_code_count"))
    )

    q = (
        versions.alias("v")
        .join(analytics.alias("a"), "model_ver_sid", "inner")
        .filter(_qa_true(F.col("a.model_ver_prom_expl_flag")))
        .select(
            "model_ver_sid",
            F.col("v.model_sid").alias("model_sid"),
            F.col("v.model_ver_num").alias("model_ver_num"),
            F.col("v.model_ver_name").alias("model_ver_name"),
            F.col("v.model_ver_signfcnt_ctgry_code").alias(
                "model_ver_signfcnt_ctgry_code"
            ),
            F.col("v.model_ver_signfcnt_lvl_name").alias(
                "model_ver_signfcnt_lvl_name"
            ),
            F.col("a.model_ver_signfcnt_ctgry_code").alias(
                "__qa_anlt_significance"
            ),
            F.col("v.model_ver_owner_block_name").alias(
                "model_ver_owner_block_name"
            ),
            F.col("v.model_ver_owner_dprtmt_name").alias(
                "model_ver_owner_dprtmt_name"
            ),
            F.col("v.model_ver_owner_cont_name").alias(
                "model_ver_owner_cont_name"
            ),
            F.col("v.model_ver_dev_block_name").alias(
                "model_ver_dev_block_name"
            ),
            F.col("v.model_ver_dev_dprtmt_name").alias(
                "model_ver_dev_dprtmt_name"
            ),
            F.col("v.model_ver_dev_dprtmt_cont_name").alias(
                "model_ver_dev_dprtmt_cont_name"
            ),
            F.col("v.model_ver_dev_start_fact_dttm").alias(
                "model_ver_dev_start_fact_dttm"
            ),
            F.col("v.model_ver_dev_end_fact_dttm").alias(
                "model_ver_dev_end_fact_dttm"
            ),
            F.col("v.model_ver_dev_report_sid").alias("model_ver_dev_report_sid"),
            F.col("v.model_ver_repstry_link_txt").alias(
                "model_ver_repstry_link_txt"
            ),
            F.col("v.model_ver_mlstorage_branch_link_txt").alias(
                "model_ver_mlstorage_branch_link_txt"
            ),
            F.col("v.model_ver_montrg_last_rslt_name").alias(
                "model_ver_montrg_last_rslt_name"
            ),
            F.col("v.model_ver_montrg_last_sid").alias(
                "model_ver_montrg_last_sid"
            ),
            F.col("a.model_ver_not_valid_flag").alias(
                "model_ver_not_valid_flag"
            ),
            F.col("a.model_ver_busn_proc_code").alias("model_ver_busn_proc_code"),
            F.col("a.model_ver_prom_actual_implm_first_dttm").alias(
                "expl_start_dttm"
            ),
            F.col("a.model_ver_valid_last_rslt_name").alias(
                "model_ver_valid_last_rslt_name"
            ),
        )
        .join(models, "model_sid", "left")
        .withColumn("__qa_code", F.upper(F.trim(F.col("model_code"))))
        .join(duplicate_codes, "__qa_code", "left")
    )

    # ---------- validation ----------
    validations = _qa_event(
        spark,
        "t_valid",
        ["valid_sid"],
        [
            "model_ver_sid",
            "valid_rslt_name",
            "valid_end_fact_dttm",
            "valid_not_sample_data_flag",
        ],
        as_of,
    ).filter(_qa_required(F.col("model_ver_sid")))

    completed_validations = validations.filter(
        _qa_required(F.col("valid_end_fact_dttm"))
        & (
            F.col("valid_end_fact_dttm").cast("date")
            <= F.lit(as_of).cast("date")
        )
    )

    samples = _qa_event(
        spark,
        "t_sample_data",
        ["sample_data_sid"],
        [
            "model_ver_sid",
            "valid_sid",
            "montrg_manual_rslt_sid",
            "montrg_auto_rslt_sid",
            "sample_data_type_name",
            "sample_data_addtnl_info_txt",
        ],
        as_of,
    )

    sample_by_valid = (
        samples.filter(
            _qa_required(F.col("valid_sid"))
            & _qa_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("valid_sid")
        .dropDuplicates(["valid_sid"])
        .withColumn("__qa_valid_sample", F.lit(1))
    )

    sample_validation_by_version = (
        samples.filter(
            _qa_required(F.col("model_ver_sid"))
            & _qa_norm(F.col("sample_data_type_name")).contains("валидац")
            & _qa_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("model_ver_sid")
        .dropDuplicates(["model_ver_sid"])
        .withColumn("__qa_version_valid_sample", F.lit(1))
    )

    validation_agg = (
        completed_validations
        .join(sample_by_valid, "valid_sid", "left")
        .join(sample_validation_by_version, "model_ver_sid", "left")
        .withColumn(
            "__qa_has_result",
            _qa_required(F.col("valid_rslt_name")).cast("int"),
        )
        .withColumn(
            "__qa_data_ok",
            (
                _qa_required(F.col("valid_rslt_name"))
                & (
                    _qa_true(F.col("valid_not_sample_data_flag"))
                    | (
                        F.coalesce(F.col("__qa_valid_sample"), F.lit(0))
                        == 1
                    )
                    | (
                        F.coalesce(
                            F.col("__qa_version_valid_sample"),
                            F.lit(0),
                        )
                        == 1
                    )
                )
            ).cast("int"),
        )
        .groupBy("model_ver_sid")
        .agg(
            F.max("__qa_has_result").alias("__qa_validation_result"),
            F.max("__qa_data_ok").alias("__qa_validation_data"),
        )
    )

    q = (
        q.join(validation_agg, "model_ver_sid", "left")
        .join(sample_validation_by_version, "model_ver_sid", "left")
        .fillna(
            {
                "__qa_validation_result": 0,
                "__qa_validation_data": 0,
                "__qa_version_valid_sample": 0,
            }
        )
        .withColumn(
            "__qa_validation_not_required",
            _qa_true(F.col("model_ver_not_valid_flag")).cast("int"),
        )
        .withColumn(
            "__qa_has_validation_result",
            (
                (F.col("__qa_validation_result") == 1)
                | _qa_required(F.col("model_ver_valid_last_rslt_name"))
            ).cast("int"),
        )
        .withColumn(
            "__qa_has_validation_data",
            (
                (F.col("__qa_validation_data") == 1)
                | (F.col("__qa_version_valid_sample") == 1)
            ).cast("int"),
        )
    )

    # ---------- monitoring ----------
    auto_results = _qa_event(
        spark,
        "t_montrg_auto_rslt",
        ["montrg_auto_rslt_sid"],
        [
            "montrg_auto_sid",
            "montrg_auto_rslt_name",
            "montrg_auto_rslt_valid_not_sample_data_flag",
        ],
        as_of,
    ).filter(_qa_required(F.col("montrg_auto_sid")))

    manual_results = _qa_event(
        spark,
        "t_montrg_manual_rslt",
        ["montrg_manual_rslt_sid"],
        [
            "montrg_manual_sid",
            "montrg_manual_rslt_name",
            "montrg_manual_rslt_valid_not_sample_data_flag",
        ],
        as_of,
    ).filter(_qa_required(F.col("montrg_manual_sid")))

    auto_links = _qa_event(
        spark,
        "t_model_ver_x_montrg_auto",
        ["model_ver_sid", "montrg_auto_sid"],
        [],
        as_of,
    )

    manual_links = _qa_event(
        spark,
        "t_model_ver_x_montrg_manual",
        ["model_ver_sid", "montrg_manual_sid"],
        [],
        as_of,
    )

    auto_sample = (
        samples.filter(
            _qa_required(F.col("montrg_auto_rslt_sid"))
            & _qa_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("montrg_auto_rslt_sid")
        .dropDuplicates(["montrg_auto_rslt_sid"])
        .withColumn("__qa_auto_sample", F.lit(1))
    )

    manual_sample = (
        samples.filter(
            _qa_required(F.col("montrg_manual_rslt_sid"))
            & _qa_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("montrg_manual_rslt_sid")
        .dropDuplicates(["montrg_manual_rslt_sid"])
        .withColumn("__qa_manual_sample", F.lit(1))
    )

    auto_data = (
        auto_results
        .join(auto_sample, "montrg_auto_rslt_sid", "left")
        .filter(_qa_required(F.col("montrg_auto_rslt_name")))
        .withColumn(
            "__qa_auto_data_ok",
            (
                _qa_true(
                    F.col("montrg_auto_rslt_valid_not_sample_data_flag")
                )
                | (F.coalesce(F.col("__qa_auto_sample"), F.lit(0)) == 1)
            ).cast("int"),
        )
        .groupBy("montrg_auto_sid")
        .agg(F.max("__qa_auto_data_ok").alias("__qa_auto_data_ok"))
    )

    manual_data = (
        manual_results
        .join(manual_sample, "montrg_manual_rslt_sid", "left")
        .filter(_qa_required(F.col("montrg_manual_rslt_name")))
        .withColumn(
            "__qa_manual_data_ok",
            (
                _qa_true(
                    F.col("montrg_manual_rslt_valid_not_sample_data_flag")
                )
                | (F.coalesce(F.col("__qa_manual_sample"), F.lit(0)) == 1)
            ).cast("int"),
        )
        .groupBy("montrg_manual_sid")
        .agg(F.max("__qa_manual_data_ok").alias("__qa_manual_data_ok"))
    )

    auto_by_version = (
        auto_links
        .join(auto_data, "montrg_auto_sid", "left")
        .select(
            F.col("model_ver_sid").alias("__qa_auto_model_ver_sid"),
            F.col("montrg_auto_sid").alias("__qa_auto_monitor_sid"),
            F.coalesce(F.col("__qa_auto_data_ok"), F.lit(0)).alias(
                "__qa_auto_data_ok"
            ),
        )
    )

    manual_by_version = (
        manual_links
        .join(manual_data, "montrg_manual_sid", "left")
        .select(
            F.col("model_ver_sid").alias("__qa_manual_model_ver_sid"),
            F.col("montrg_manual_sid").alias("__qa_manual_monitor_sid"),
            F.coalesce(F.col("__qa_manual_data_ok"), F.lit(0)).alias(
                "__qa_manual_data_ok"
            ),
        )
    )

    version_monitor_sample = (
        samples.filter(
            _qa_required(F.col("model_ver_sid"))
            & _qa_norm(F.col("sample_data_type_name")).rlike(
                r"монитор|стабил"
            )
            & _qa_location(F.col("sample_data_addtnl_info_txt"))
        )
        .select("model_ver_sid")
        .dropDuplicates(["model_ver_sid"])
        .withColumn("__qa_version_monitor_sample", F.lit(1))
    )

    q = (
        q.join(
            auto_by_version,
            (
                F.col("model_ver_sid")
                == F.col("__qa_auto_model_ver_sid")
            )
            & (
                F.col("model_ver_montrg_last_sid")
                == F.col("__qa_auto_monitor_sid")
            ),
            "left",
        )
        .join(
            manual_by_version,
            (
                F.col("model_ver_sid")
                == F.col("__qa_manual_model_ver_sid")
            )
            & (
                F.col("model_ver_montrg_last_sid")
                == F.col("__qa_manual_monitor_sid")
            ),
            "left",
        )
        .join(version_monitor_sample, "model_ver_sid", "left")
        .withColumn(
            "__qa_has_monitor_data",
            (
                (F.coalesce(F.col("__qa_auto_data_ok"), F.lit(0)) == 1)
                | (
                    F.coalesce(
                        F.col("__qa_manual_data_ok"),
                        F.lit(0),
                    )
                    == 1
                )
                | (
                    F.coalesce(
                        F.col("__qa_version_monitor_sample"),
                        F.lit(0),
                    )
                    == 1
                )
            ).cast("int"),
        )
    )

    # ---------- independent expected flags ----------
    q = (
        q
        .withColumn("QA_F01_MODEL_CODE", (~_qa_required(F.col("model_code"))).cast("int"))
        .withColumn("QA_F02_MODEL_NAME", (~_qa_required(F.col("model_name"))).cast("int"))
        .withColumn(
            "QA_F03_MODEL_CODE_DUP",
            (F.coalesce(F.col("__qa_code_count"), F.lit(0)) > 1).cast("int"),
        )
        .withColumn("QA_F04_VERSION_NUM", (~_qa_required(F.col("model_ver_num"))).cast("int"))
        .withColumn("QA_F05_VERSION_NAME", (~_qa_required(F.col("model_ver_name"))).cast("int"))
        .withColumn(
            "QA_F06_DEV_START",
            (~_qa_required(F.col("model_ver_dev_start_fact_dttm"))).cast("int"),
        )
        .withColumn(
            "QA_F07_DEV_END",
            (~_qa_required(F.col("model_ver_dev_end_fact_dttm"))).cast("int"),
        )
        .withColumn(
            "QA_F08_EXPL_START",
            (~_qa_required(F.col("expl_start_dttm"))).cast("int"),
        )
        .withColumn(
            "QA_F09_OWNER",
            (
                (~_qa_required(F.col("model_ver_owner_block_name")))
                | (~_qa_required(F.col("model_ver_owner_dprtmt_name")))
                | (~_qa_required(F.col("model_ver_owner_cont_name")))
            ).cast("int"),
        )
        .withColumn(
            "QA_F10_DEVELOPER",
            (
                (~_qa_required(F.col("model_ver_dev_block_name")))
                | (~_qa_required(F.col("model_ver_dev_dprtmt_name")))
                | (~_qa_required(F.col("model_ver_dev_dprtmt_cont_name")))
            ).cast("int"),
        )
        .withColumn(
            "QA_F11_BUSN_PROCESS",
            F.when(_qa_is_na(F.col("model_ver_busn_proc_code")), F.lit(0))
            .otherwise((~_qa_filled(F.col("model_ver_busn_proc_code"))).cast("int")),
        )
        .withColumn(
            "QA_F12_SIGNIFICANCE",
            (~_qa_significance(F.col("model_ver_signfcnt_ctgry_code"))).cast("int"),
        )
        .withColumn(
            "QA_F13_DEV_REPORT",
            (~_qa_required(F.col("model_ver_dev_report_sid"))).cast("int"),
        )
        .withColumn(
            "QA_F14_BITBUCKET_CODE",
            (~(
                _qa_bitbucket(F.col("model_ver_repstry_link_txt"))
                | _qa_bitbucket(F.col("model_ver_mlstorage_branch_link_txt"))
            )).cast("int"),
        )
        .withColumn(
            "QA_F15_VALIDATION_RESULT",
            (
                (F.col("__qa_validation_not_required") == 0)
                & (F.col("__qa_has_validation_result") == 0)
            ).cast("int"),
        )
        .withColumn(
            "QA_F16_MONITOR_RESULT",
            (
                _qa_required(F.col("model_ver_montrg_last_sid"))
                & (~_qa_required(F.col("model_ver_montrg_last_rslt_name")))
            ).cast("int"),
        )
        .withColumn(
            "QA_F17_VALIDATION_DATA",
            (
                (F.col("__qa_validation_not_required") == 0)
                & (F.col("__qa_has_validation_result") == 1)
                & (F.col("__qa_has_validation_data") == 0)
            ).cast("int"),
        )
        .withColumn(
            "QA_F18_MONITOR_DATA",
            (
                _qa_required(F.col("model_ver_montrg_last_sid"))
                & _qa_required(F.col("model_ver_montrg_last_rslt_name"))
                & (F.col("__qa_has_monitor_data") == 0)
            ).cast("int"),
        )
        .withColumn(
            "QA_F19_SIGNIFICANCE_MISMATCH",
            (
                _qa_significance(F.col("model_ver_signfcnt_ctgry_code"))
                & _qa_significance(F.col("__qa_anlt_significance"))
                & (
                    _qa_norm(F.col("model_ver_signfcnt_ctgry_code"))
                    != _qa_norm(F.col("__qa_anlt_significance"))
                )
            ).cast("int"),
        )
    )

    select_cols = ["model_ver_sid"]
    select_cols += [
        F.col(c).alias("QA_" + c)
        for c in QA_INFO_COLUMNS
    ]
    select_cols += ["QA_" + c for c in FLAG_COLUMNS]

    return q.select(*select_cols)


def run_qa(spark, report, as_of, fail_on_mismatch=True):
    """
    Сверяет основной отчёт с независимым контрольным расчётом.
    Экспорт должен выполняться только после успешного QA.
    """
    control = build_qa_control(spark, as_of).cache()

    try:
        report_ids = report.select("model_ver_sid").dropDuplicates()
        control_ids = control.select("model_ver_sid").dropDuplicates()

        report_count = report_ids.count()
        control_count = control_ids.count()

        missing_from_report = control_ids.join(
            report_ids,
            "model_ver_sid",
            "left_anti",
        ).cache()

        unexpected_in_report = report_ids.join(
            control_ids,
            "model_ver_sid",
            "left_anti",
        ).cache()

        missing_count = missing_from_report.count()
        unexpected_count = unexpected_in_report.count()

        joined = (
            report.alias("r")
            .join(control.alias("q"), "model_ver_sid", "inner")
        )

        comparison_specs = []

        for name in QA_INFO_COLUMNS:
            comparison_specs.append(
                (
                    "VALUE:" + name,
                    F.col("r." + name).cast("string"),
                    F.col("q.QA_" + name).cast("string"),
                )
            )

        for flag in FLAG_COLUMNS:
            comparison_specs.append(
                (
                    "FLAG:" + flag,
                    F.col("r." + flag).cast("string"),
                    F.col("q.QA_" + flag).cast("string"),
                )
            )

        mismatch_array = F.array(
            *[
                F.struct(
                    F.lit(check_name).alias("check_name"),
                    actual.alias("actual"),
                    expected.alias("expected"),
                )
                for check_name, actual, expected in comparison_specs
            ]
        )

        mismatches = (
            joined
            .select(
                "model_ver_sid",
                F.explode(mismatch_array).alias("__qa_cmp"),
            )
            .select(
                "model_ver_sid",
                F.col("__qa_cmp.check_name").alias("check_name"),
                F.col("__qa_cmp.actual").alias("actual"),
                F.col("__qa_cmp.expected").alias("expected"),
            )
            .filter(
                ~F.col("actual").eqNullSafe(F.col("expected"))
            )
            .cache()
        )

        mismatch_count = mismatches.count()

        # Независимая сверка распределения A-E, включая NULL/прочие значения.
        report_dist = (
            report
            .withColumn(
                "__qa_category",
                F.coalesce(
                    F.col("model_ver_signfcnt_ctgry_code").cast("string"),
                    F.lit("<NULL>"),
                ),
            )
            .groupBy("__qa_category")
            .count()
            .withColumnRenamed("count", "report_count")
        )

        control_dist = (
            control
            .withColumn(
                "__qa_category",
                F.coalesce(
                    F.col("QA_model_ver_signfcnt_ctgry_code").cast("string"),
                    F.lit("<NULL>"),
                ),
            )
            .groupBy("__qa_category")
            .count()
            .withColumnRenamed("count", "control_count")
        )

        distribution = (
            report_dist.join(
                control_dist,
                "__qa_category",
                "full",
            )
            .fillna({"report_count": 0, "control_count": 0})
            .orderBy("__qa_category")
            .cache()
        )

        distribution_mismatch = distribution.filter(
            F.col("report_count") != F.col("control_count")
        ).count()

        summary = {
            "status": "OK",
            "report_rows": report_count,
            "control_rows": control_count,
            "missing_from_report": missing_count,
            "unexpected_in_report": unexpected_count,
            "value_or_flag_mismatches": mismatch_count,
            "distribution_mismatches": distribution_mismatch,
        }

        failed = (
            report_count != control_count
            or missing_count > 0
            or unexpected_count > 0
            or mismatch_count > 0
            or distribution_mismatch > 0
        )

        if failed:
            summary["status"] = "FAILED"

        print("\nQA / self-check")
        print("-" * 60)
        for key, value in summary.items():
            print(f"{key}: {value}")

        print("\nРаспределение степени значимости:")
        distribution.show(20, truncate=False)

        if missing_count:
            print("\nЕсть в контрольном периметре, но отсутствуют в отчёте:")
            missing_from_report.show(QA_SHOW_MISMATCHES, truncate=False)

        if unexpected_count:
            print("\nЕсть в отчёте, но отсутствуют в контрольном периметре:")
            unexpected_in_report.show(QA_SHOW_MISMATCHES, truncate=False)

        if mismatch_count:
            print("\nРасхождения значений/флагов:")
            mismatches.orderBy("model_ver_sid", "check_name").show(
                QA_SHOW_MISMATCHES,
                truncate=False,
            )

            print("\nКоличество расхождений по проверкам:")
            mismatches.groupBy("check_name").count().orderBy(
                F.desc("count"),
                "check_name",
            ).show(100, truncate=False)

        if failed and fail_on_mismatch:
            raise RuntimeError(
                "QA FAILED: основной отчёт расходится с независимым "
                "контрольным расчётом. Excel не сохранён."
            )

        return summary

    finally:
        for frame_name in (
            "missing_from_report",
            "unexpected_in_report",
            "mismatches",
            "distribution",
        ):
            frame = locals().get(frame_name)
            if frame is not None:
                try:
                    frame.unpersist()
                except Exception:
                    pass
        control.unpersist()

## 8. Run / main

In [ ]:
FLAG_COLUMNS = [
    "F01_MODEL_CODE",
    "F02_MODEL_NAME",
    "F03_MODEL_CODE_DUP",
    "F04_VERSION_NUM",
    "F05_VERSION_NAME",
    "F06_DEV_START",
    "F07_DEV_END",
    "F08_EXPL_START",
    "F09_OWNER",
    "F10_DEVELOPER",
    "F11_BUSN_PROCESS",
    "F12_SIGNIFICANCE",
    "F13_DEV_REPORT",
    "F14_BITBUCKET_CODE",
    "F15_VALIDATION_RESULT",
    "F16_MONITOR_RESULT",
    "F17_VALIDATION_DATA",
    "F18_MONITOR_DATA",
    "F19_SIGNIFICANCE_MISMATCH",
]


def validate_report(report):
    """Проверяет структурные инварианты результата перед выгрузкой."""
    if report.filter(~required_filled(F.col("model_ver_sid"))).limit(1).count():
        raise RuntimeError("В отчёте обнаружен пустой/служебный model_ver_sid")

    duplicate = (
        report.groupBy("model_ver_sid")
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )
    if duplicate:
        raise RuntimeError("В отчёте больше одной строки на model_ver_sid")

    invalid_flag = reduce(
        lambda a, b: a | b,
        [
            F.col(c).isNull() | (~F.col(c).isin(0, 1))
            for c in FLAG_COLUMNS
        ],
        F.lit(False),
    )
    if report.filter(invalid_flag).limit(1).count():
        raise RuntimeError("Один или несколько флагов содержат значение не из {0, 1}")

    expected_issue_count = reduce(
        lambda a, b: a + b,
        [F.col(c) for c in FLAG_COLUMNS],
        F.lit(0),
    )
    if report.filter(F.col("issue_count") != expected_issue_count).limit(1).count():
        raise RuntimeError("issue_count не совпадает с суммой флагов")


def run(
    spark,
    as_of,
    output_file=None,
    save_excel=True,
    show_rows=20,
):
    report = build_report(spark, as_of).cache()

    try:
        validate_report(report)
        version_count = report.count()
        print("Версий в эксплуатации:", version_count)

        qa_summary = run_qa(
            spark=spark,
            report=report,
            as_of=as_of,
            fail_on_mismatch=QA_FAIL_ON_MISMATCH,
        )

        if show_rows:
            report.show(show_rows, truncate=False)

        if save_excel:
            if output_file is None:
                raise ValueError("Для save_excel=True необходимо задать output_file")
            write_excel(report, output_file)
            print("Сохранено:", Path(output_file).resolve())

        return report, qa_summary

    except Exception:
        report.unpersist()
        raise


def main():
    return run(
        spark=spark,
        as_of=AS_OF,
        output_file=OUTPUT_FILE,
        save_excel=True,
        show_rows=20,
    )

## 9. Запуск

In [ ]:
# При повторном запуске освобождаем кэш предыдущего отчёта.
if "report" in globals():
    try:
        report.unpersist()
    except Exception:
        pass

report, qa_summary = main()


## 10. Точечная диагностика

In [ ]:
# Итог последнего QA:
# qa_summary

# Пример проверки конкретной версии:
# report.filter(F.col("model_ver_sid") == "123145").show(truncate=False)

# Только строки с нарушениями:
# report.filter(F.col("issue_count") > 0).show(50, truncate=False)

# Повторный QA без остановки на расхождениях:
# qa_summary = run_qa(spark, report, AS_OF, fail_on_mismatch=False)

# После завершения работы:
# report.unpersist()
